# Silver to Gold load

Builds the Gold star schema from the Silver tables: 7 dimensions, 5 fact tables, 7 aggregate tables and 4 audit views for Power BI.
It is the third task of the job (after `bronze_to_silver`). Dimensions and facts are written with `MERGE INTO`; aggregates are overwritten because they are small and derived from the facts.

**Standard run:** Run all with the default widgets. Only Silver rows whose `load_timestamp` is newer than the last successful Gold run are read for the Arena fact. Everything else is rebuilt from Silver and merged by content hash, so an unchanged row is never rewritten.
**Backfill:** `since_timestamp` (process Silver rows changed since that time) or `full_refresh=true` (rebuild from all Silver rows).
**One part only:** `table_group` = `dimensions`, `facts`, `aggregates` or `views`.

### CELL 1 - Parameters and configuration

In [ ]:
# CELL 1 - Parameters and configuration
import json, re, traceback, uuid, difflib
from datetime import date, datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (BooleanType, DoubleType, LongType, StringType, StructField, StructType,
                               TimestampType, IntegerType)

dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("silver_schema", "silver", "silver_schema")
dbutils.widgets.text("gold_schema", "gold", "gold_schema")
dbutils.widgets.dropdown("table_group", "all", ["all", "dimensions", "facts", "aggregates", "views"], "table_group")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"], "full_refresh: rebuild from all Silver rows")
dbutils.widgets.text("since_timestamp", "", "since_timestamp (UTC, e.g. 2026-10-01T00:00:00): backfill from this Silver load time")
dbutils.widgets.dropdown("optimize", "true", ["true", "false"], "optimize: run OPTIMIZE on fact tables that changed")

P = {k: dbutils.widgets.get(k).strip() for k in
     ["catalog", "silver_schema", "gold_schema", "table_group", "full_refresh", "since_timestamp", "optimize"]}
PARAM_ERRORS = []
SINCE = None
if P["since_timestamp"]:
    try:
        SINCE = datetime.fromisoformat(P["since_timestamp"].replace("Z", "")).strftime("%Y-%m-%d %H:%M:%S")
    except ValueError:
        PARAM_ERRORS.append(f"since_timestamp='{P['since_timestamp']}' is not a valid ISO timestamp")
FULL = P["full_refresh"] == "true"
GROUP = P["table_group"]
CATALOG, SSCHEMA, GSCHEMA = P["catalog"], P["silver_schema"], P["gold_schema"]
for _name in (CATALOG, SSCHEMA, GSCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", _name):
        raise ValueError(f"Invalid catalog/schema name: {_name!r}")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
spark.conf.set("spark.sql.session.timeZone", "UTC")
S = lambda n: f"{CATALOG}.{SSCHEMA}.{n}"
G = lambda n: f"{CATALOG}.{GSCHEMA}.{n}"
T_RUNS, T_ERRORS = f"{CATALOG}.ops.pipeline_execution_logs", f"{CATALOG}.ops.error_logs"
LAYER = "Silver-to-Gold"
print("Run id:", RUN_ID, "| group:", GROUP, "| full_refresh:", FULL, "| since:", SINCE, "| parameters:", P)

### CELL 2 - The Gold contracts: every table, column, type and primary key is written out here

In [ ]:
# CELL 2 - The Gold contracts

def q(name):
    return "`" + name.replace("`", "``") + "`"

TABLES = {}
def reg(name, kind, cols, keys, cluster=None):
    """kind: dim, fact or scd2 (merged, with row_hash) and agg (overwritten)."""
    TABLES[name] = dict(kind=kind, cols=cols, keys=keys, cluster=cluster or [])

reg("dim_date", "dim", [("date_key", "INT"), ("date", "DATE"), ("year", "INT"), ("quarter", "INT"), ("month", "INT"),
    ("month_name", "STRING"), ("year_month", "STRING"), ("week_of_year", "INT"), ("day_of_week", "INT"),
    ("day_name", "STRING"), ("is_weekend", "BOOLEAN")], ["date_key"])
reg("dim_model", "dim", [("model_sk", "BIGINT"), ("canonical_model_key", "STRING"), ("base_model_key", "STRING"),
    ("base_model_sk", "BIGINT"), ("base_match_method", "STRING"), ("display_name", "STRING"), ("organization", "STRING"),
    ("family", "STRING"), ("country", "STRING"), ("license", "STRING"), ("release_date", "DATE"),
    ("release_date_key", "INT"), ("parameters", "DOUBLE"), ("open_model_weights", "BOOLEAN"), ("in_arena", "BOOLEAN"),
    ("in_benchlm", "BOOLEAN"), ("in_pricing", "BOOLEAN"), ("in_epoch", "BOOLEAN"), ("in_cooperbench", "BOOLEAN"),
    ("sources_count", "INT"), ("evidence_coverage", "STRING")], ["model_sk"])
reg("dim_arena_category", "dim", [("category_sk", "BIGINT"), ("subset", "STRING"), ("category", "STRING"),
    ("category_group", "STRING"), ("style_controlled", "BOOLEAN"), ("display_name", "STRING"), ("sort_order", "INT")],
    ["category_sk"])
reg("dim_capability", "dim", [("capability_sk", "INT"), ("capability_key", "STRING"), ("display_name", "STRING"),
    ("sort_order", "INT")], ["capability_sk"])
reg("dim_study_setting", "dim", [("setting_sk", "INT"), ("study_setting", "STRING"), ("setting_label", "STRING"),
    ("is_cooperative", "BOOLEAN"), ("sort_order", "INT")], ["setting_sk"])
reg("dim_task", "dim", [("task_sk", "BIGINT"), ("repo", "STRING"), ("task_id", "BIGINT"), ("feature_pair", "STRING"),
    ("feature_a", "INT"), ("feature_b", "INT")], ["task_sk"])
reg("dim_workload_scenario", "dim", [("scenario_sk", "INT"), ("scenario_name", "STRING"),
    ("input_mtok_per_month", "DOUBLE"), ("output_mtok_per_month", "DOUBLE"), ("description", "STRING")],
    ["scenario_sk"])

reg("fact_arena_rating", "fact", [("model_sk", "BIGINT"), ("category_sk", "BIGINT"), ("date_key", "INT"),
    ("rating", "DOUBLE"), ("rating_ci_lower", "DOUBLE"), ("rating_ci_upper", "DOUBLE"), ("rating_variance", "DOUBLE"),
    ("vote_count", "BIGINT"), ("source_rank", "BIGINT"), ("rating_rank", "INT"), ("models_in_group", "INT"),
    ("rating_percentile", "DOUBLE"), ("silver_load_timestamp", "TIMESTAMP")],
    ["model_sk", "category_sk", "date_key"], cluster=["date_key", "category_sk"])
reg("fact_model_price", "scd2", [("model_sk", "BIGINT"), ("valid_from_ts", "TIMESTAMP"), ("valid_to_ts", "TIMESTAMP"),
    ("valid_from_date_key", "INT"), ("is_current", "BOOLEAN"), ("input_price_per_mtok", "DOUBLE"),
    ("output_price_per_mtok", "DOUBLE"), ("cached_input_price_per_mtok", "DOUBLE"), ("training_price", "DOUBLE"),
    ("context_window_tokens", "BIGINT"), ("has_numeric_pricing", "BOOLEAN"), ("is_free_pricing", "BOOLEAN"),
    ("price_hash", "STRING")], ["model_sk", "valid_from_ts"])
reg("fact_model_release", "fact", [("model_sk", "BIGINT"), ("publication_date_key", "INT"), ("parameters", "DOUBLE"),
    ("training_compute_flop", "DOUBLE"), ("training_dataset_size_total", "DOUBLE"), ("training_time_hours", "DOUBLE"),
    ("hardware_quantity", "DOUBLE"), ("training_compute_cost_usd_2023", "DOUBLE"), ("training_power_draw_w", "DOUBLE"),
    ("citations", "BIGINT"), ("frontier_model", "STRING"), ("foundation_model", "STRING"),
    ("open_model_weights", "BOOLEAN")], ["model_sk", "publication_date_key"])
reg("fact_capability_score", "fact", [("model_sk", "BIGINT"), ("capability_sk", "INT"), ("snapshot_date_key", "INT"),
    ("score", "DOUBLE"), ("verified_score", "DOUBLE")], ["model_sk", "capability_sk"])
reg("fact_agent_task_result", "fact", [("setting_sk", "INT"), ("task_sk", "BIGINT"), ("agent_model_sk", "BIGINT"),
    ("evaluated_date_key", "INT"), ("has_eval", "BOOLEAN"), ("has_run", "BOOLEAN"), ("both_passed", "BOOLEAN"),
    ("feature1_passed", "BOOLEAN"), ("feature2_passed", "BOOLEAN"), ("feature1_tests_passed", "BIGINT"),
    ("feature1_tests_failed", "BIGINT"), ("feature2_tests_passed", "BIGINT"), ("feature2_tests_failed", "BIGINT"),
    ("merge_status", "STRING"), ("merge_strategy", "STRING"), ("merge_conflict", "BOOLEAN"),
    ("apply_status_agent1", "STRING"), ("apply_status_agent2", "STRING"), ("duration_seconds", "DOUBLE"),
    ("total_steps", "BIGINT"), ("messages_sent", "BIGINT"), ("messages_agent1", "BIGINT"),
    ("messages_agent2", "BIGINT"), ("total_cost", "DOUBLE")], ["setting_sk", "task_sk"])

reg("agg_arena_latest", "agg", [("model_sk", "BIGINT"), ("category_sk", "BIGINT"), ("date_key", "INT"),
    ("rating", "DOUBLE"), ("rating_ci_lower", "DOUBLE"), ("rating_ci_upper", "DOUBLE"), ("vote_count", "BIGINT"),
    ("rating_rank", "INT"), ("rating_percentile", "DOUBLE"), ("is_on_latest_board", "BOOLEAN")],
    ["model_sk", "category_sk"])
reg("agg_style_sensitivity", "agg", [("model_sk", "BIGINT"), ("category", "STRING"), ("rating_default", "DOUBLE"),
    ("rating_style_control", "DOUBLE"), ("rank_default", "INT"), ("rank_style_control", "INT"),
    ("rank_shift", "INT")], ["model_sk", "category"])
reg("agg_frontier_timeline", "agg", [("category_sk", "BIGINT"), ("date_key", "INT"), ("top_rating", "DOUBLE"),
    ("top_model_sk", "BIGINT"), ("models_ranked", "BIGINT"), ("best_rating_so_far", "DOUBLE"),
    ("is_new_leader", "BOOLEAN")], ["category_sk", "date_key"])
reg("agg_capability_cost", "agg", [("model_sk", "BIGINT"), ("scenario_sk", "INT"), ("arena_rating", "DOUBLE"),
    ("arena_rating_style_control", "DOUBLE"), ("benchlm_display_score", "DOUBLE"),
    ("input_price_per_mtok", "DOUBLE"), ("output_price_per_mtok", "DOUBLE"), ("est_monthly_cost_usd", "DOUBLE"),
    ("rating_points_per_dollar", "DOUBLE")], ["model_sk", "scenario_sk"])
reg("agg_coordination_by_setting", "agg", [("setting_sk", "INT"), ("tasks", "BIGINT"), ("evaluated", "BIGINT"),
    ("passed", "BIGINT"), ("pass_rate", "DOUBLE"), ("avg_duration_seconds", "DOUBLE"), ("merge_conflicts", "BIGINT"),
    ("merge_conflict_rate", "DOUBLE"), ("evaluation_coverage", "DOUBLE")], ["setting_sk"])
reg("agg_coordination_paired", "agg", [("task_sk", "BIGINT"), ("passed_solo", "BOOLEAN"),
    ("passed_coopgit", "BOOLEAN"), ("passed_team", "BOOLEAN"), ("passed_team_noproto", "BOOLEAN"),
    ("coopgit_vs_solo", "STRING"), ("team_vs_solo", "STRING"), ("team_noproto_vs_solo", "STRING")], ["task_sk"])
reg("model_alias", "agg", [("source", "STRING"), ("source_key", "STRING"), ("base_key", "STRING"),
    ("method", "STRING")], ["source", "source_key"])

GROUPS = {
    "dimensions": ["dim_date", "dim_model", "dim_arena_category", "dim_capability", "dim_study_setting", "dim_task",
                   "dim_workload_scenario"],
    "facts": ["fact_arena_rating", "fact_model_price", "fact_model_release", "fact_capability_score",
              "fact_agent_task_result"],
    "aggregates": ["agg_arena_latest", "agg_style_sensitivity", "agg_frontier_timeline", "agg_capability_cost",
                   "agg_coordination_by_setting", "agg_coordination_paired"],
}
# (fact table, foreign key column, dimension table, dimension key). Nullable keys are checked only when present.
FOREIGN_KEYS = [
    ("fact_arena_rating", "model_sk", "dim_model", "model_sk"), ("fact_arena_rating", "category_sk", "dim_arena_category", "category_sk"),
    ("fact_arena_rating", "date_key", "dim_date", "date_key"), ("fact_model_price", "model_sk", "dim_model", "model_sk"),
    ("fact_model_price", "valid_from_date_key", "dim_date", "date_key"), ("fact_model_release", "model_sk", "dim_model", "model_sk"),
    ("fact_model_release", "publication_date_key", "dim_date", "date_key"), ("fact_capability_score", "model_sk", "dim_model", "model_sk"),
    ("fact_capability_score", "capability_sk", "dim_capability", "capability_sk"),
    ("fact_agent_task_result", "setting_sk", "dim_study_setting", "setting_sk"), ("fact_agent_task_result", "task_sk", "dim_task", "task_sk"),
    ("fact_agent_task_result", "agent_model_sk", "dim_model", "model_sk"), ("fact_agent_task_result", "evaluated_date_key", "dim_date", "date_key"),
]

# Reviewed model identity overrides: {arena model key: base model key}. Use None to refuse a rule-based match.
# Add a line here after reviewing the alias review cell, then run the notebook again.
ALIAS_OVERRIDES = {}

### CELL 3 - Logging helpers (same ops tables as Bronze and Silver)

In [ ]:
# CELL 3 - Logging helpers
OPS_TYPES = {"string": StringType(), "bigint": LongType(), "double": DoubleType(), "boolean": BooleanType(),
             "timestamp": TimestampType()}
OPS_SPECS = {
    T_RUNS: [("run_id", "string"), ("batch_id", "string"), ("layer", "string"), ("source", "string"),
             ("dataset", "string"), ("load_type", "string"), ("file_or_param", "string"),
             ("parameters_json", "string"), ("start_time", "timestamp"), ("end_time", "timestamp"),
             ("duration_seconds", "double"), ("status", "string"), ("rows_read", "bigint"),
             ("rows_inserted", "bigint"), ("rows_updated", "bigint"), ("rows_skipped_duplicate", "bigint"),
             ("rows_quarantined", "bigint"), ("bytes_processed", "bigint"), ("message", "string"),
             ("load_timestamp", "timestamp")],
    T_ERRORS: [("error_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("component", "string"),
               ("layer", "string"), ("source", "string"), ("file_or_param", "string"), ("severity", "string"),
               ("error_type", "string"), ("error_message", "string"), ("http_status", "bigint"),
               ("attempt", "bigint"), ("stack_trace", "string"), ("occurred_at", "timestamp"),
               ("resolved", "boolean"), ("load_timestamp", "timestamp")],
}
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ops")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GSCHEMA}")
for _table, _spec in OPS_SPECS.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {_table} ({', '.join(q(n) + ' ' + t.upper() for n, t in _spec)}) USING DELTA")


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def append_record(table, record):
    spec = OPS_SPECS[table]
    schema = StructType([StructField(n, StringType() if t == "timestamp" else OPS_TYPES[t], True) for n, t in spec])
    df = spark.createDataFrame([tuple(record.get(n) for n, _ in spec)], schema)
    for name, kind in spec:
        if kind == "timestamp":
            df = df.withColumn(name, F.to_timestamp(F.col(name)))
    df.write.mode("append").saveAsTable(table)


class Counts:
    def __init__(self):
        self.read = self.inserted = self.updated = self.skipped = self.quarantined = 0


def log_run(layer, file_or_param, dataset, start, status, c=None, message="", params=None):
    end = now_iso()
    c = c or Counts()
    duration = (datetime.strptime(end, "%Y-%m-%dT%H:%M:%S.%fZ")
                - datetime.strptime(start, "%Y-%m-%dT%H:%M:%S.%fZ")).total_seconds()
    append_record(T_RUNS, {
        "run_id": RUN_ID, "batch_id": None, "layer": layer, "source": "gold", "dataset": dataset,
        "load_type": None, "file_or_param": file_or_param,
        "parameters_json": json.dumps(params if params is not None else P), "start_time": start, "end_time": end,
        "duration_seconds": duration, "status": status, "rows_read": c.read, "rows_inserted": c.inserted,
        "rows_updated": c.updated, "rows_skipped_duplicate": c.skipped, "rows_quarantined": c.quarantined,
        "bytes_processed": None, "message": str(message)[:4000], "load_timestamp": now_iso()})


def log_error(file_or_param, message, error_type="", stack="", severity="ERROR"):
    print(f"{severity} | {file_or_param} | {message}")
    try:
        append_record(T_ERRORS, {
            "error_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": None, "component": "gold_notebook",
            "layer": LAYER, "source": "gold", "file_or_param": file_or_param, "severity": severity,
            "error_type": error_type, "error_message": str(message)[:4000], "http_status": None, "attempt": None,
            "stack_trace": stack[-4000:], "occurred_at": now_iso(), "resolved": False, "load_timestamp": now_iso()})
    except Exception as exc:
        print("Could not write to the error table:", exc)


def last_watermark(name):
    """The Silver load_timestamp that the last successful run of this table had reached."""
    rows = (spark.table(T_RUNS).filter((F.col("layer") == LAYER) & (F.col("status") == "SUCCESS")
                                       & (F.col("file_or_param") == f"{GSCHEMA}.{name}"))
            .orderBy(F.col("end_time").desc()).select("message").limit(1).collect())
    if not rows or not rows[0][0]:
        return None
    try:
        return json.loads(rows[0][0]).get("silver_watermark")
    except ValueError:
        return None

### CELL 4 - Builders: pure functions from Silver DataFrames to Gold DataFrames (no table access, so they can be tested alone)

In [ ]:
# CELL 4 - Builders

def date_key(c):
    return F.date_format(F.col(c) if isinstance(c, str) else c, "yyyyMMdd").cast("int")


def model_key_expr(col):
    """Same rule as Silver: lower case, runs of other characters become a hyphen."""
    return F.regexp_replace(F.regexp_replace(F.lower(col), "[^a-z0-9]+", "-"), "^-+|-+$", "")


def model_sk_of(col):
    return F.xxhash64(col)


# ------------------------------------------------------------------ seed dimensions
def build_dim_date():
    d = spark.sql("SELECT explode(sequence(to_date('1950-01-01'), to_date('2030-12-31'), interval 1 day)) AS date")
    return d.select(date_key("date").alias("date_key"), "date", F.year("date").alias("year"),
                    F.quarter("date").alias("quarter"), F.month("date").alias("month"),
                    F.date_format("date", "MMMM").alias("month_name"), F.date_format("date", "yyyy-MM").alias("year_month"),
                    F.weekofyear("date").alias("week_of_year"), F.dayofweek("date").alias("day_of_week"),
                    F.date_format("date", "EEEE").alias("day_name"), F.dayofweek("date").isin(1, 7).alias("is_weekend"))


CAPABILITIES = [(0, "overall", "Overall"), (1, "agentic", "Agentic"), (2, "coding", "Coding"),
                (3, "instructionFollowing", "Instruction following"), (4, "knowledge", "Knowledge"),
                (5, "math", "Math"), (6, "multilingual", "Multilingual"),
                (7, "multimodalGrounded", "Multimodal grounded"), (8, "reasoning", "Reasoning")]
SETTINGS = [(1, "solo", "Single agent", False), (2, "coopgit", "Shared Git", True), (3, "team", "Full team", True),
            (4, "team_noproto", "Team without protocol", True)]
SCENARIOS = [(1, "Light assistant", 5.0, 1.0, "5M input and 1M output tokens a month, for example a small helper tool"),
             (2, "Team assistant", 50.0, 10.0, "50M input and 10M output tokens a month, a shared assistant for a team"),
             (3, "Heavy coding agent", 500.0, 150.0, "500M input and 150M output tokens a month, agents running all day")]


def build_dim_capability():
    return spark.createDataFrame([(a, b, c, a + 1) for a, b, c in CAPABILITIES],
                                 "capability_sk INT, capability_key STRING, display_name STRING, sort_order INT")


def build_dim_study_setting():
    return spark.createDataFrame([(a, b, c, d, a) for a, b, c, d in SETTINGS],
                                 "setting_sk INT, study_setting STRING, setting_label STRING, is_cooperative BOOLEAN, sort_order INT")


def build_dim_workload_scenario():
    return spark.createDataFrame(SCENARIOS, "scenario_sk INT, scenario_name STRING, input_mtok_per_month DOUBLE, "
                                            "output_mtok_per_month DOUBLE, description STRING")


# ------------------------------------------------------------------ dimensions from Silver
LANGUAGES = ["english", "chinese", "german", "french", "spanish", "japanese", "korean", "russian", "polish", "non_english"]


def build_dim_arena_category(arena):
    c = arena.select("subset", "category").where(F.col("subset").isNotNull() & F.col("category").isNotNull()).distinct()
    group = (F.when(F.col("category") == "overall", "Overall")
             .when(F.col("category").startswith("industry_"), "Industry")
             .when((F.col("subset") == "webdev") | F.col("category").startswith("webdev"), "Web development")
             .when(F.col("subset") == "agent", "Agent")
             .when(F.col("category").isin(*LANGUAGES), "Language").otherwise("Capability"))
    order = (F.when(group == "Overall", 1).when(group == "Capability", 2).when(group == "Language", 3)
             .when(group == "Industry", 4).when(group == "Web development", 5).otherwise(6))
    c = c.select("subset", "category", group.alias("category_group"), order.alias("_g"),
                 (F.col("subset") == "text_style_control").alias("style_controlled"))
    label = F.initcap(F.regexp_replace(F.regexp_replace("category", "^industry_", ""), "_", " "))
    c = c.withColumn("display_name", F.concat(label, F.when(F.col("style_controlled"), F.lit(" (style control)")).otherwise(F.lit(""))))
    w = Window.orderBy("_g", "subset", "category")
    return c.select(F.xxhash64("subset", "category").alias("category_sk"), "subset", "category", "category_group",
                    "style_controlled", "display_name", F.row_number().over(w).cast("int").alias("sort_order"))


def build_dim_task(evals, runs):
    t = (evals.select("repo", "task_id", "feature_pair").unionByName(runs.select("repo", "task_id", "feature_pair"))
         .where(F.col("repo").isNotNull() & F.col("task_id").isNotNull() & F.col("feature_pair").isNotNull()).distinct())
    t = (t.withColumn("_a", F.regexp_extract("feature_pair", r"f(\d+)_f(\d+)", 1))
          .withColumn("_b", F.regexp_extract("feature_pair", r"f(\d+)_f(\d+)", 2)))
    return t.select(F.xxhash64("repo", F.col("task_id").cast("string"), "feature_pair").alias("task_sk"), "repo",
                    "task_id", "feature_pair", F.expr("try_cast(nullif(_a, '') AS INT)").alias("feature_a"),
                    F.expr("try_cast(nullif(_b, '') AS INT)").alias("feature_b"))


# ------------------------------------------------------------------ model identity
STRIP_WORDS = {"high", "xhigh", "max", "medium", "low", "minimal", "thinking", "reasoning", "think"}


def _strippable(tok):
    return (tok in STRIP_WORDS or re.fullmatch(r"\d{8}", tok) is not None or re.fullmatch(r"\d{1,3}k", tok) is not None
            or re.fullmatch(r"(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])", tok) is not None)


def derive_base(key, external, arena_keys):
    """Map an Arena configuration key (for example claude-opus-5-5-high) to the model that the other sources
    know (claude-opus-5-5). Only trailing effort, budget and date tokens are removed, and only when the
    shorter key really exists. Anything else stays unmatched, so versions are never merged by guesswork."""
    if key in external:
        return key, "EXACT"
    toks = key.split("-")
    work = list(toks)
    while len(work) > 1 and _strippable(work[-1]):
        work.pop()
        cand = "-".join(work)
        if cand in external:
            return cand, "RULE_STRIP_SUFFIX"
    work = list(toks)
    while len(work) > 1 and _strippable(work[-1]):
        work.pop()
        cand = "-".join(work)
        if cand in arena_keys:
            return cand, "RULE_ARENA_GROUP"
    return key, "UNMATCHED"


def build_alias(arena_keys, external_keys, overrides):
    """Returns a list of (source, source_key, base_key, method) rows for the Arena model keys."""
    ext, ar = set(external_keys), set(arena_keys)
    rows = []
    for k in sorted(ar):
        base, method = derive_base(k, ext, ar - {k})
        if k in overrides:
            if overrides[k] is None:
                base, method = k, "OVERRIDE_REFUSED"
            else:
                base, method = overrides[k], "OVERRIDE"
        rows.append(("arena", k, base, method))
    return rows


def build_dim_model(arena, benchlm, pricing, epoch, coop_runs, alias_rows):
    alias = spark.createDataFrame(alias_rows, "source STRING, source_key STRING, base_key STRING, method STRING")
    newest = F.struct("leaderboard_publish_date", F.coalesce("vote_count", F.lit(0)), "model_name")
    a = (arena.groupBy("model_key").agg(F.max_by("model_name", newest).alias("a_name"),
                                         F.max_by("organization", newest).alias("a_org"),
                                         F.max_by("license", newest).alias("a_license")))
    b = (benchlm.where(F.col("model_key").isNotNull()).groupBy("model_key")
         .agg(F.max_by("model", "collection_timestamp").alias("b_name"), F.max_by("creator", "collection_timestamp").alias("b_creator"),
              F.max_by("family", "collection_timestamp").alias("b_family"), F.max_by("release_date", "collection_timestamp").alias("b_release")))
    p = (pricing.where(F.col("model_key").isNotNull()).groupBy("model_key")
         .agg(F.max_by("model", "collection_timestamp").alias("p_name"), F.max_by("creator", "collection_timestamp").alias("p_creator")))
    latest_e = F.struct("publication_date", "last_modified", "model")
    e = (epoch.where(F.col("model_key").isNotNull()).groupBy("model_key")
         .agg(F.max_by("model", latest_e).alias("e_name"), F.max_by("organization", latest_e).alias("e_org"),
              F.max_by("country_of_organization", latest_e).alias("e_country"),
              F.max_by("parameters", latest_e).alias("e_params"),
              F.max_by("open_model_weights", latest_e).alias("e_open"), F.max("publication_date").alias("e_pub")))
    c = (coop_runs.where(F.col("model").isNotNull()).select(model_key_expr(F.col("model")).alias("model_key"),
                                                            F.col("model").alias("c_name")).distinct())
    keys = (a.select("model_key").unionByName(b.select("model_key")).unionByName(p.select("model_key"))
            .unionByName(e.select("model_key")).unionByName(c.select("model_key")).where(F.col("model_key").isNotNull())
            .distinct())
    base = keys.join(alias.select(F.col("source_key").alias("model_key"), F.col("base_key").alias("_base"),
                                  F.col("method").alias("_method")), "model_key", "left")
    base = base.select("model_key", F.coalesce("_base", "model_key").alias("base_model_key"),
                       F.coalesce("_method", F.lit("SELF")).alias("base_match_method"))
    ak = a.select("model_key", F.lit(True).alias("in_arena"), "a_name", "a_org", "a_license")
    ck = c.select("model_key", F.lit(True).alias("in_cooperbench"), "c_name")
    bb = b.select(F.col("model_key").alias("base_model_key"), F.lit(True).alias("in_benchlm"), "b_name", "b_creator", "b_family", "b_release")
    pb = p.select(F.col("model_key").alias("base_model_key"), F.lit(True).alias("in_pricing"), "p_name", "p_creator")
    eb = e.select(F.col("model_key").alias("base_model_key"), F.lit(True).alias("in_epoch"), "e_name", "e_org", "e_country", "e_params", "e_open", "e_pub")
    j = (base.join(ak, "model_key", "left").join(ck, "model_key", "left").join(bb, "base_model_key", "left")
         .join(pb, "base_model_key", "left").join(eb, "base_model_key", "left"))
    flag = lambda n: F.coalesce(F.col(n), F.lit(False))
    j = j.select("model_key", "base_model_key", "base_match_method", flag("in_arena").alias("in_arena"),
                 flag("in_benchlm").alias("in_benchlm"), flag("in_pricing").alias("in_pricing"),
                 flag("in_epoch").alias("in_epoch"), flag("in_cooperbench").alias("in_cooperbench"),
                 F.coalesce("a_name", "b_name", "p_name", "e_name", "c_name", "model_key").alias("display_name"),
                 F.coalesce("b_creator", "p_creator", F.initcap("a_org"), "e_org").alias("organization"),
                 F.col("b_family").alias("family"), F.col("e_country").alias("country"), F.col("a_license").alias("license"),
                 F.coalesce("e_pub", "b_release").alias("release_date"), F.col("e_params").alias("parameters"),
                 F.col("e_open").alias("open_model_weights"))
    cov = F.concat_ws(" + ", F.when(F.col("in_arena"), "Arena"), F.when(F.col("in_benchlm"), "BenchLM"),
                      F.when(F.col("in_pricing"), "Pricing"), F.when(F.col("in_epoch"), "Epoch"),
                      F.when(F.col("in_cooperbench"), "CooperBench"))
    return j.select(F.xxhash64("model_key").alias("model_sk"), F.col("model_key").alias("canonical_model_key"),
                    "base_model_key", F.xxhash64("base_model_key").alias("base_model_sk"), "base_match_method",
                    "display_name", "organization", "family", "country", "license", "release_date",
                    date_key("release_date").alias("release_date_key"), "parameters", "open_model_weights",
                    "in_arena", "in_benchlm", "in_pricing", "in_epoch", "in_cooperbench",
                    (F.col("in_arena").cast("int") + F.col("in_benchlm").cast("int") + F.col("in_pricing").cast("int")
                     + F.col("in_epoch").cast("int") + F.col("in_cooperbench").cast("int")).alias("sources_count"),
                    F.when(cov == "", "No source").otherwise(cov).alias("evidence_coverage"))


# ------------------------------------------------------------------ facts
def build_fact_arena(arena_groups):
    """arena_groups: every Silver Arena row of the (subset, category, date) groups that changed."""
    a = arena_groups.where(F.col("model_key").isNotNull())
    f = a.select(F.xxhash64("model_key").alias("model_sk"), F.xxhash64("subset", "category").alias("category_sk"),
                 date_key("leaderboard_publish_date").alias("date_key"), "rating", "rating_ci_lower", "rating_ci_upper",
                 "rating_variance", "vote_count", F.col("rank").alias("source_rank"),
                 F.col("load_timestamp").alias("silver_load_timestamp"))
    # two model names can share one model key: keep the entry with more votes
    pick = Window.partitionBy("model_sk", "category_sk", "date_key").orderBy(F.col("vote_count").desc_nulls_last(),
                                                                         F.col("rating").desc())
    f = f.withColumn("_n", F.row_number().over(pick)).where("_n = 1").drop("_n")
    grp = Window.partitionBy("category_sk", "date_key")
    return (f.withColumn("rating_rank", F.dense_rank().over(grp.orderBy(F.col("rating").desc())).cast("int"))
             .withColumn("models_in_group", F.count(F.lit(1)).over(grp).cast("int"))
             .withColumn("rating_percentile", F.when(F.col("models_in_group") == 1, F.lit(100.0)).otherwise(
                 F.round(F.percent_rank().over(grp.orderBy(F.col("rating").asc())) * 100, 2))))


PRICE_COLS = ["input_price", "output_price", "cached_input_price", "training_price"]


def build_price_source(pricing):
    p = pricing.where(F.col("model_key").isNotNull())
    num = F.col("has_numeric_pricing")
    s = p.select(F.xxhash64("model_key").alias("model_sk"), F.col("collection_timestamp").alias("valid_from_ts"),
                 F.when(num, F.col("input_price")).alias("input_price_per_mtok"),
                 F.when(num, F.col("output_price")).alias("output_price_per_mtok"),
                 F.when(num, F.col("cached_input_price")).alias("cached_input_price_per_mtok"),
                 F.when(num, F.col("training_price")).alias("training_price"), "context_window_tokens",
                 "has_numeric_pricing", "is_free_pricing")
    pick = Window.partitionBy("model_sk").orderBy(F.col("valid_from_ts").desc())
    s = s.withColumn("_n", F.row_number().over(pick)).where("_n = 1").drop("_n")
    cols = ["input_price_per_mtok", "output_price_per_mtok", "cached_input_price_per_mtok", "training_price",
            "context_window_tokens", "has_numeric_pricing", "is_free_pricing"]
    h = F.sha2(F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u2205")) for c in cols]), 256)
    return (s.withColumn("price_hash", h).withColumn("valid_to_ts", F.lit(None).cast("timestamp"))
             .withColumn("valid_from_date_key", date_key(F.col("valid_from_ts").cast("date")))
             .withColumn("is_current", F.lit(True)))


def build_fact_release(epoch):
    e = epoch.where(F.col("model_key").isNotNull() & F.col("publication_date").isNotNull())
    f = e.select(F.xxhash64("model_key").alias("model_sk"), date_key("publication_date").alias("publication_date_key"),
                 "parameters", "training_compute_flop", "training_dataset_size_total", "training_time_hours",
                 "hardware_quantity", "training_compute_cost_usd_2023", "training_power_draw_w", "citations",
                 "frontier_model", "foundation_model", "open_model_weights", "last_modified")
    pick = Window.partitionBy("model_sk", "publication_date_key").orderBy(F.col("last_modified").desc_nulls_last())
    return f.withColumn("_n", F.row_number().over(pick)).where("_n = 1").drop("_n", "last_modified")


CAP_KEYS = ["agentic", "coding", "instructionFollowing", "knowledge", "math", "multilingual", "multimodalGrounded", "reasoning"]
CAP_SCHEMA = StructType([
    StructField("displayCategoryScores", StructType([StructField(k, DoubleType()) for k in CAP_KEYS])),
    StructField("verifiedDisplayCategoryScores", StructType([StructField(k, DoubleType()) for k in CAP_KEYS])),
    StructField("verifiedDisplayScore", DoubleType())])


def build_fact_capability(benchlm):
    b = benchlm.where(F.col("model_key").isNotNull() & F.col("display_score").isNotNull())
    p = b.select("model_key", "display_score", "collection_timestamp", F.from_json("scores_json", CAP_SCHEMA).alias("s"))
    parts = [F.struct(F.lit(k).alias("capability_key"), F.col("s.displayCategoryScores." + k).alias("score"),
                      F.col("s.verifiedDisplayCategoryScores." + k).alias("verified")) for k in CAP_KEYS]
    parts.append(F.struct(F.lit("overall").alias("capability_key"), F.col("display_score").alias("score"),
                          F.col("s.verifiedDisplayScore").alias("verified")))
    x = p.select("model_key", "collection_timestamp", F.explode(F.array(*parts)).alias("c")).select(
        "model_key", "collection_timestamp", F.col("c.capability_key").alias("capability_key"),
        F.col("c.score").alias("score"), F.col("c.verified").alias("verified_score")).where(F.col("score").isNotNull())
    cap = build_dim_capability().select("capability_sk", "capability_key")
    x = x.join(cap, "capability_key").select(F.xxhash64("model_key").alias("model_sk"), "capability_sk",
                                              date_key(F.col("collection_timestamp").cast("date")).alias("snapshot_date_key"),
                                              "score", "verified_score", "collection_timestamp")
    pick = Window.partitionBy("model_sk", "capability_sk").orderBy(F.col("collection_timestamp").desc())
    return x.withColumn("_n", F.row_number().over(pick)).where("_n = 1").drop("_n", "collection_timestamp")


def build_fact_agent(evals, runs, members):
    keys = ["study_setting", "repo", "task_id", "feature_pair"]
    ok = lambda d: d.where(F.col("repo").isNotNull() & F.col("task_id").isNotNull() & F.col("feature_pair").isNotNull())
    e = ok(evals).select(*keys, F.lit(True).alias("has_eval"), "both_passed", "feature1_passed", "feature2_passed",
                         "feature1_tests_passed", "feature1_tests_failed", "feature2_tests_passed", "feature2_tests_failed",
                         "merge_status", "merge_strategy", "apply_status_agent1", "apply_status_agent2",
                         F.to_date("evaluated_at").alias("evaluated_date"))
    r = ok(runs).select(*keys, F.lit(True).alias("has_run"), "duration_seconds", "total_steps", "messages_sent",
                        "total_cost", F.col("model").alias("agent_model"))
    m = (ok(members.where(F.col("file_type") == "agent_sent")).groupBy(*keys)
         .agg(F.max(F.when(F.col("file_name").startswith("agent1"), F.col("line_count"))).alias("messages_agent1"),
              F.max(F.when(F.col("file_name").startswith("agent2"), F.col("line_count"))).alias("messages_agent2")))
    j = e.join(r, keys, "full_outer").join(m, keys, "left")
    sett = build_dim_study_setting().select("setting_sk", "study_setting")
    j = j.join(sett, "study_setting", "left")
    return j.select(F.col("setting_sk"), F.xxhash64("repo", F.col("task_id").cast("string"), "feature_pair").alias("task_sk"),
                    F.when(F.col("agent_model").isNotNull(), F.xxhash64(model_key_expr(F.col("agent_model")))).alias("agent_model_sk"),
                    date_key("evaluated_date").alias("evaluated_date_key"),
                    F.coalesce("has_eval", F.lit(False)).alias("has_eval"), F.coalesce("has_run", F.lit(False)).alias("has_run"),
                    "both_passed", "feature1_passed", "feature2_passed", "feature1_tests_passed", "feature1_tests_failed",
                    "feature2_tests_passed", "feature2_tests_failed", "merge_status", "merge_strategy",
                    F.when(F.col("merge_status").isNotNull(), F.col("merge_status") == "conflicts").alias("merge_conflict"),
                    "apply_status_agent1", "apply_status_agent2", "duration_seconds", "total_steps", "messages_sent",
                    "messages_agent1", "messages_agent2",
                    F.when(F.col("total_cost") > 0, F.col("total_cost")).alias("total_cost")).where(F.col("setting_sk").isNotNull())


# ------------------------------------------------------------------ aggregates (built from Gold tables)
def build_agg_arena_latest(fact):
    w = Window.partitionBy("model_sk", "category_sk").orderBy(F.col("date_key").desc())
    latest = fact.withColumn("_n", F.row_number().over(w)).where("_n = 1").drop("_n")
    latest = latest.withColumn("_maxd", F.max("date_key").over(Window.partitionBy("category_sk")))
    return latest.select("model_sk", "category_sk", "date_key", "rating", "rating_ci_lower", "rating_ci_upper",
                         "vote_count", "rating_rank", "rating_percentile",
                         (F.col("date_key") == F.col("_maxd")).alias("is_on_latest_board"))


def build_agg_style(latest, dim_cat):
    c = dim_cat.select("category_sk", "subset", "category").where(F.col("subset").isin("text", "text_style_control"))
    x = latest.where("is_on_latest_board").join(c, "category_sk")
    d = x.where(F.col("subset") == "text").select("model_sk", "category", F.col("rating").alias("rating_default"),
                                                   F.col("rating_rank").alias("rank_default"))
    s = x.where(F.col("subset") == "text_style_control").select("model_sk", "category", F.col("rating").alias("rating_style_control"),
                                                                 F.col("rating_rank").alias("rank_style_control"))
    return d.join(s, ["model_sk", "category"]).withColumn("rank_shift", (F.col("rank_default") - F.col("rank_style_control")).cast("int"))


def build_agg_frontier(fact):
    g = fact.groupBy("category_sk", "date_key").agg(F.max("rating").alias("top_rating"),
                                                    F.max_by("model_sk", "rating").alias("top_model_sk"),
                                                    F.count(F.lit(1)).alias("models_ranked"))
    w = Window.partitionBy("category_sk").orderBy("date_key")
    return (g.withColumn("best_rating_so_far", F.max("top_rating").over(w.rowsBetween(Window.unboundedPreceding, 0)))
             .withColumn("_prev", F.lag("top_model_sk").over(w))
             .withColumn("is_new_leader", F.col("_prev").isNull() | (F.col("_prev") != F.col("top_model_sk"))).drop("_prev"))


def build_agg_cost(latest, dim_model, dim_cat, price, cap):
    cat = lambda subset: dim_cat.where((F.col("subset") == subset) & (F.col("category") == "overall")).select("category_sk")
    base = lambda subset, name: (latest.where("is_on_latest_board").join(cat(subset), "category_sk")
                                 .select("model_sk", F.col("rating").alias(name)))
    r = base("text", "arena_rating").join(base("text_style_control", "arena_rating_style_control"), "model_sk", "left")
    r = r.join(dim_model.select("model_sk", "base_model_sk"), "model_sk")
    cur = price.where("is_current").select(F.col("model_sk").alias("base_model_sk"), "input_price_per_mtok", "output_price_per_mtok")
    r = r.join(cur, "base_model_sk")
    score = cap.where(F.col("capability_sk") == 0).select(F.col("model_sk").alias("base_model_sk"),
                                                           F.col("score").alias("benchlm_display_score"))
    r = r.join(score, "base_model_sk", "left").crossJoin(build_dim_workload_scenario())
    cost = F.col("input_price_per_mtok") * F.col("input_mtok_per_month") + F.col("output_price_per_mtok") * F.col("output_mtok_per_month")
    r = r.withColumn("est_monthly_cost_usd", cost).where(F.col("est_monthly_cost_usd").isNotNull())
    return r.select("model_sk", "scenario_sk", "arena_rating", "arena_rating_style_control", "benchlm_display_score",
                    "input_price_per_mtok", "output_price_per_mtok", "est_monthly_cost_usd",
                    F.when(F.col("est_monthly_cost_usd") > 0, F.col("arena_rating") / F.col("est_monthly_cost_usd")).alias("rating_points_per_dollar"))


def build_agg_coord_setting(fact):
    g = fact.groupBy("setting_sk").agg(
        F.count(F.lit(1)).alias("tasks"), F.sum(F.col("has_eval").cast("int")).alias("evaluated"),
        F.sum(F.when(F.col("both_passed"), 1).otherwise(0)).alias("passed"),
        F.avg("duration_seconds").alias("avg_duration_seconds"),
        F.sum(F.when(F.col("merge_conflict"), 1).otherwise(0)).alias("merge_conflicts"))
    return g.select("setting_sk", "tasks", "evaluated", "passed",
                    F.round(F.col("passed") / F.col("evaluated"), 4).alias("pass_rate"), "avg_duration_seconds", "merge_conflicts",
                    F.round(F.col("merge_conflicts") / F.col("evaluated"), 4).alias("merge_conflict_rate"),
                    F.round(F.col("evaluated") / F.col("tasks"), 4).alias("evaluation_coverage"))


def build_agg_coord_paired(fact):
    s = build_dim_study_setting().select("setting_sk", "study_setting")
    x = fact.join(s, "setting_sk").where(F.col("has_eval"))
    piv = lambda name: F.max(F.when(F.col("study_setting") == name, F.col("both_passed")))
    p = x.groupBy("task_sk").agg(piv("solo").alias("passed_solo"), piv("coopgit").alias("passed_coopgit"),
                                  piv("team").alias("passed_team"), piv("team_noproto").alias("passed_team_noproto"))
    def vs(col):
        return (F.when(F.col("passed_solo").isNull() | F.col(col).isNull(), "not comparable")
                 .when(F.col("passed_solo") & F.col(col), "both passed")
                 .when(F.col("passed_solo") & ~F.col(col), "only single agent")
                 .when(~F.col("passed_solo") & F.col(col), "only cooperative").otherwise("neither"))
    return (p.withColumn("coopgit_vs_solo", vs("passed_coopgit")).withColumn("team_vs_solo", vs("passed_team"))
             .withColumn("team_noproto_vs_solo", vs("passed_team_noproto")))

### CELL 5 - Engine: write helpers, one step per Gold table, data quality checks

In [ ]:
# CELL 5 - Engine

def ensure_table(name):
    spec = TABLES[name]
    cols = [f"{q(n)} {t}" for n, t in spec["cols"]]
    if spec["kind"] in ("dim", "fact"):
        cols.append("`row_hash` STRING")
    cols.append("`load_timestamp` TIMESTAMP")
    cl = f" CLUSTER BY ({', '.join(q(c) for c in spec['cluster'])})" if spec["cluster"] else ""
    spark.sql(f"CREATE TABLE IF NOT EXISTS {G(name)} ({', '.join(cols)}) USING DELTA{cl}")


def typed(name, df):
    spec = TABLES[name]
    return df.select(*[F.col(n).cast(t.lower()).alias(n) for n, t in spec["cols"]])


def merge_table(name, df, stage=False):
    """Upsert on the primary key. A row is updated only when its content hash changed.
    stage=True writes the prepared rows to a temporary table first, so a heavy transformation (windows, joins)
    is computed once and the count and the MERGE both read the result."""
    spec = TABLES[name]
    names, keys = [n for n, _ in spec["cols"]], spec["keys"]
    nonkey = [n for n in names if n not in keys]
    s = typed(name, df).withColumn("row_hash", F.sha2(F.concat_ws("\u0001", *[
        F.coalesce(F.col(n).cast("string"), F.lit("\u2205")) for n in nonkey]), 256)).withColumn("load_timestamp", F.current_timestamp())
    if stage:
        stg = G("_stg_" + name)
        s.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(stg)
        s = spark.table(stg)
    n_source = s.count()
    view = "gold_src_" + uuid.uuid4().hex[:8]
    s.createOrReplaceTempView(view)
    on = " AND ".join(f"t.{q(k)} = s.{q(k)}" for k in keys)
    sets = ", ".join(f"{q(c)} = s.{q(c)}" for c in nonkey + ["row_hash", "load_timestamp"])
    allc = names + ["row_hash", "load_timestamp"]
    res = spark.sql(f"""MERGE INTO {G(name)} AS t USING {view} AS s ON {on}
        WHEN MATCHED AND NOT (t.row_hash <=> s.row_hash) THEN UPDATE SET {sets}
        WHEN NOT MATCHED THEN INSERT ({', '.join(q(c) for c in allc)}) VALUES ({', '.join('s.' + q(c) for c in allc)})"""
                    ).collect()[0].asDict()
    spark.catalog.dropTempView(view)
    if stage:
        spark.sql(f"DROP TABLE IF EXISTS {stg}")
    return int(res.get("num_inserted_rows") or 0), int(res.get("num_updated_rows") or 0), n_source


def overwrite_table(name, df):
    out = typed(name, df).withColumn("load_timestamp", F.current_timestamp())
    out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(G(name))
    return spark.table(G(name)).count()


def scd2_price(src):
    """Type 2 history: a changed price closes the current row and inserts a new version."""
    view = "gold_price_" + uuid.uuid4().hex[:8]
    typed("fact_model_price", src).createOrReplaceTempView(view)
    closed = spark.sql(f"""MERGE INTO {G('fact_model_price')} AS t USING {view} AS s
        ON t.model_sk = s.model_sk AND t.is_current
        WHEN MATCHED AND t.price_hash <> s.price_hash
          THEN UPDATE SET is_current = false, valid_to_ts = s.valid_from_ts, load_timestamp = current_timestamp()"""
                       ).collect()[0].asDict()
    names = [n for n, _ in TABLES["fact_model_price"]["cols"]]
    ins = spark.sql(f"""INSERT INTO {G('fact_model_price')}
        SELECT {', '.join('s.' + q(n) for n in names)}, current_timestamp()
        FROM {view} s LEFT ANTI JOIN {G('fact_model_price')} c
          ON c.model_sk = s.model_sk AND c.is_current AND c.price_hash = s.price_hash""").collect()[0].asDict()
    spark.catalog.dropTempView(view)
    return int(ins.get("num_inserted_rows") or 0), int(closed.get("num_updated_rows") or 0)


sil = lambda n: spark.table(S(n))
CHANGED, _ALIAS = {}, {}


def alias_rows():
    if "rows" not in _ALIAS:
        arena_keys = [r[0] for r in sil("arena_leaderboard").select("model_key").where(F.col("model_key").isNotNull()).distinct().collect()]
        ext = set()
        for t in ("benchlm_models", "benchlm_pricing", "epoch_models"):
            ext |= {r[0] for r in sil(t).select("model_key").where(F.col("model_key").isNotNull()).distinct().collect()}
        _ALIAS["rows"] = build_alias(arena_keys, ext, ALIAS_OVERRIDES)
    return _ALIAS["rows"]


STEPS = {}
def step(name):
    def deco(fn):
        STEPS[name] = fn
        return fn
    return deco


def simple_merge(name, builder):
    def fn(c, msg):
        ins, upd, n = merge_table(name, builder())
        c.read, c.inserted, c.updated, c.skipped = n, ins, upd, n - ins - upd
        CHANGED[name] = (ins + upd) > 0
    STEPS[name] = fn


def simple_overwrite(name, builder):
    def fn(c, msg):
        n = overwrite_table(name, builder())
        c.read = c.inserted = n
        CHANGED[name] = True
    STEPS[name] = fn


gt = lambda n: spark.table(G(n))
simple_merge("dim_date", build_dim_date)
simple_merge("dim_capability", build_dim_capability)
simple_merge("dim_study_setting", build_dim_study_setting)
simple_merge("dim_workload_scenario", build_dim_workload_scenario)
simple_merge("dim_arena_category", lambda: build_dim_arena_category(sil("arena_leaderboard")))
simple_merge("dim_task", lambda: build_dim_task(sil("cooperbench_evals"), sil("cooperbench_runs")))
simple_merge("dim_model", lambda: build_dim_model(sil("arena_leaderboard"), sil("benchlm_models"), sil("benchlm_pricing"),
                                                  sil("epoch_models"), sil("cooperbench_runs"), alias_rows()))
simple_overwrite("model_alias", lambda: spark.createDataFrame(alias_rows(), "source STRING, source_key STRING, base_key STRING, method STRING"))
simple_merge("fact_model_release", lambda: build_fact_release(sil("epoch_models")))
simple_merge("fact_capability_score", lambda: build_fact_capability(sil("benchlm_models")))
simple_merge("fact_agent_task_result", lambda: build_fact_agent(sil("cooperbench_evals"), sil("cooperbench_runs"), sil("cooperbench_members")))


@step("fact_model_price")
def _price(c, msg):
    src = build_price_source(sil("benchlm_pricing"))
    n = src.count()
    ins, closed = scd2_price(src)
    c.read, c.inserted, c.updated, c.skipped = n, ins, closed, n - ins
    CHANGED["fact_model_price"] = (ins + closed) > 0
    msg["new_versions"], msg["closed_versions"] = ins, closed


@step("fact_arena_rating")
def _arena(c, msg):
    arena = sil("arena_leaderboard")
    wm = None if FULL else (SINCE or last_watermark("fact_arena_rating"))
    changed = arena if wm is None else arena.where(F.col("load_timestamp") > F.lit(wm).cast("timestamp"))
    new_wm = changed.agg(F.max("load_timestamp")).collect()[0][0]
    msg["silver_watermark_in"] = wm
    if new_wm is None:
        msg["note"], msg["silver_watermark"] = "no changed Silver rows", wm
        CHANGED["fact_arena_rating"] = False
        return
    groups = changed.select("subset", "category", "leaderboard_publish_date").distinct()
    scope = arena.join(groups, ["subset", "category", "leaderboard_publish_date"], "left_semi")
    ins, upd, n = merge_table("fact_arena_rating", build_fact_arena(scope), stage=True)
    c.read, c.inserted, c.updated, c.skipped = n, ins, upd, n - ins - upd
    CHANGED["fact_arena_rating"] = (ins + upd) > 0
    msg["silver_watermark"] = new_wm.strftime("%Y-%m-%d %H:%M:%S.%f")


simple_overwrite("agg_arena_latest", lambda: build_agg_arena_latest(gt("fact_arena_rating")))
simple_overwrite("agg_style_sensitivity", lambda: build_agg_style(gt("agg_arena_latest"), gt("dim_arena_category")))
simple_overwrite("agg_frontier_timeline", lambda: build_agg_frontier(gt("fact_arena_rating")))
simple_overwrite("agg_capability_cost", lambda: build_agg_cost(gt("agg_arena_latest"), gt("dim_model"), gt("dim_arena_category"),
                                                               gt("fact_model_price"), gt("fact_capability_score")))
simple_overwrite("agg_coordination_by_setting", lambda: build_agg_coord_setting(gt("fact_agent_task_result")))
simple_overwrite("agg_coordination_paired", lambda: build_agg_coord_paired(gt("fact_agent_task_result")))

# ---------------------------------------------------------------- audit views for Power BI
OPS = lambda n: f"{CATALOG}.ops.{n}"
VIEWS = {
    "v_pipeline_run": f"""SELECT run_id, batch_id, layer,
        CASE WHEN layer LIKE 'Source%' THEN 'Collect' WHEN layer LIKE 'Raw-to-Bronze%' OR layer LIKE 'Bronze data quality%' THEN 'Bronze'
             WHEN layer LIKE 'Bronze-to-Silver%' OR layer LIKE 'Silver data quality%' THEN 'Silver'
             WHEN layer LIKE 'Silver-to-Gold%' OR layer LIKE 'Gold%' THEN 'Gold' ELSE 'Other' END AS stage,
        CASE WHEN layer LIKE '%(run)' THEN 'run' WHEN layer LIKE '%(batch)' THEN 'batch'
             WHEN layer LIKE '%data quality%' THEN 'quality check' ELSE 'file or table' END AS level,
        source, dataset, load_type, file_or_param, status, start_time, end_time, duration_seconds,
        ROUND(duration_seconds / 60.0, 3) AS duration_minutes, CAST(date_format(start_time, 'yyyyMMdd') AS INT) AS date_key,
        rows_read, rows_inserted, rows_updated, rows_skipped_duplicate, rows_quarantined, bytes_processed, message
        FROM {OPS('pipeline_execution_logs')}""",
    "v_quarantine": f"""SELECT 'Silver record' AS stage, silver_table AS dataset, reason, column_name AS detail_column, bad_value AS detail_value,
        batch_id, quarantined_at, CAST(date_format(quarantined_at, 'yyyyMMdd') AS INT) AS date_key FROM {OPS('silver_quarantine_log')}
        UNION ALL SELECT 'Bronze file', dataset, reason, item, detail, batch_id, quarantined_at,
        CAST(date_format(quarantined_at, 'yyyyMMdd') AS INT) FROM {OPS('quarantine_log')}""",
    "v_schema_drift": f"""SELECT drift_id, run_id, batch_id, dataset, column_name, event_type, expected_type, actual_type, action_taken,
        detected_at, CAST(date_format(detected_at, 'yyyyMMdd') AS INT) AS date_key FROM {OPS('schema_drift_log')}""",
    "v_errors": f"""SELECT error_id, run_id, batch_id, component, layer, source, file_or_param, severity, error_type, error_message,
        occurred_at, resolved, CAST(date_format(occurred_at, 'yyyyMMdd') AS INT) AS date_key FROM {OPS('error_logs')}""",
}


def make_views_step(vname):
    def fn(c, msg):
        spark.sql(f"CREATE OR REPLACE VIEW {G(vname)} AS {VIEWS[vname]}")
        c.read = spark.table(G(vname)).count()
    return fn
for _v in VIEWS:
    STEPS[_v] = make_views_step(_v)
GROUPS["views"] = list(VIEWS)
GROUPS["dimensions"] = GROUPS["dimensions"] + ["model_alias"]
ORDER = GROUPS["dimensions"] + GROUPS["facts"] + GROUPS["aggregates"] + GROUPS["views"]


def run_step(name):
    start, c, msg = now_iso(), Counts(), {}
    try:
        if name in TABLES and TABLES[name]["kind"] != "agg":
            ensure_table(name)
        STEPS[name](c, msg)
        log_run(LAYER, f"{GSCHEMA}.{name}", name, start, "SUCCESS", c, json.dumps(msg))
        print(f"OK   {GSCHEMA}.{name}: read {c.read}, inserted {c.inserted}, updated {c.updated}, skipped {c.skipped}")
        return True, c
    except Exception as exc:
        log_error(f"{GSCHEMA}.{name}", exc, type(exc).__name__, traceback.format_exc())
        log_run(LAYER, f"{GSCHEMA}.{name}", name, start, "FAILURE", c, f"{type(exc).__name__}: {exc}")
        return False, c


def check_keys(names):
    ok_all = True
    for name in names:
        start, c = now_iso(), Counts()
        try:
            df, keys = gt(name), TABLES[name]["keys"]
            c.read = df.count()
            dups = df.groupBy(*keys).count().where("count > 1").count()
            nulls = df.where(" OR ".join(f"{q(k)} IS NULL" for k in keys)).count()
            ok = dups == 0 and nulls == 0
            log_run("Gold data quality (primary key)", f"{GSCHEMA}.{name}", name, start, "SUCCESS" if ok else "FAILURE", c,
                    json.dumps({"key": keys, "rows": c.read, "duplicate_keys": dups, "null_keys": nulls}))
            print(f"KEY CHECK {name}: rows={c.read}, duplicate keys={dups}, null keys={nulls}")
            ok_all &= ok
        except Exception as exc:
            log_error(f"{GSCHEMA}.{name}", exc, type(exc).__name__, traceback.format_exc())
            ok_all = False
    return ok_all


def check_foreign_keys():
    ok_all = True
    start = now_iso()
    for fact, fk, dim, dk in FOREIGN_KEYS:
        s2, c = now_iso(), Counts()
        try:
            f = gt(fact).select(fk).where(F.col(fk).isNotNull()).distinct()
            orphans = f.join(gt(dim).select(F.col(dk).alias(fk)).distinct(), fk, "left_anti").count()
            c.read, ok = f.count(), orphans == 0
            log_run("Gold data quality (referential integrity)", f"{GSCHEMA}.{fact}.{fk} -> {dim}.{dk}", fact, s2,
                    "SUCCESS" if ok else "FAILURE", c, json.dumps({"distinct_keys": c.read, "orphan_keys": orphans}))
            print(f"FK CHECK {fact}.{fk} -> {dim}.{dk}: orphans={orphans}")
            ok_all &= ok
        except Exception as exc:
            log_error(f"{GSCHEMA}.{fact}.{fk}", exc, type(exc).__name__, traceback.format_exc())
            ok_all = False
    return ok_all

### CELL 6 - Run: dimensions, facts, aggregates, views, then the key and referential checks

In [ ]:
# CELL 6 - Run
RUN_START, total, failed_any = now_iso(), Counts(), False
if PARAM_ERRORS:
    log_run(f"{LAYER} (run)", "parameters", "all", RUN_START, "FAILURE", total, "; ".join(PARAM_ERRORS))
    raise ValueError("Bad parameters: " + "; ".join(PARAM_ERRORS))

selected = ORDER if GROUP == "all" else GROUPS[GROUP]
print(f"Tables to process: {len(selected)} ({GROUP})")
for name in selected:
    ok, c = run_step(name)
    failed_any |= not ok
    for a in ("read", "inserted", "updated", "skipped"):
        setattr(total, a, getattr(total, a) + getattr(c, a))

# keep the largest fact clustered after it changed (liquid clustering, incremental OPTIMIZE)
if P["optimize"] == "true" and CHANGED.get("fact_arena_rating"):
    o_start = now_iso()
    spark.sql(f"OPTIMIZE {G('fact_arena_rating')}")
    log_run("Gold optimize", f"{GSCHEMA}.fact_arena_rating", "fact_arena_rating", o_start, "SUCCESS", Counts(), "OPTIMIZE with liquid clustering")
    print("OPTIMIZE done: fact_arena_rating")

if GROUP in ("all", "dimensions", "facts", "aggregates"):
    ran = [n for n in selected if n in TABLES and n != "model_alias"]
    check_list = [n for n in ran if FULL or TABLES[n]["kind"] == "agg" or CHANGED.get(n)]
    failed_any |= not check_keys(check_list)
    if GROUP in ("all", "facts", "dimensions") and (FULL or any(CHANGED.get(n) for n in ran)):
        failed_any |= not check_foreign_keys()

status = "FAILURE" if failed_any else "SUCCESS"
log_run(f"{LAYER} (run)", "run", "all", RUN_START, status, total, f"tables processed: {len(selected)}")
print(f"RUN {status}: tables={len(selected)}, read={total.read}, inserted={total.inserted}, updated={total.updated}")
if failed_any:
    raise Exception("Silver-to-Gold finished with failures; see ops.error_logs and ops.pipeline_execution_logs")

### CELL 7 - Model identity review

Shows how the Arena model keys were mapped to the models the other sources know, and the top-rated Arena models that are still unmatched with the closest candidate keys.
To accept or refuse a match, add a line to `ALIAS_OVERRIDES` in Cell 2 (`"arena-key": "base-key"`, or `None` to refuse a rule), then run the notebook again.

In [ ]:
# CELL 7 - Model identity review
al = spark.table(G("model_alias"))
display(al.groupBy("method").count().orderBy(F.desc("count")))
display(al.where("method = 'RULE_STRIP_SUFFIX'").select("source_key", "base_key").orderBy("source_key").limit(60))

ext = set()
for t in ("benchlm_models", "benchlm_pricing", "epoch_models"):
    ext |= {r[0] for r in sil(t).select("model_key").where(F.col("model_key").isNotNull()).distinct().collect()}
latest = (sil("arena_leaderboard").where((F.col("subset") == "text") & (F.col("category") == "overall"))
          .where(F.col("leaderboard_publish_date") == sil("arena_leaderboard").where((F.col("subset") == "text") & (F.col("category") == "overall"))
                 .agg(F.max("leaderboard_publish_date")).collect()[0][0])
          .join(al.where("method IN ('UNMATCHED', 'RULE_ARENA_GROUP')").select(F.col("source_key").alias("model_key")), "model_key")
          .orderBy(F.col("rating").desc()).limit(40).select("model_key", "model_name", "organization", "rating").collect())
rows = [(r["model_key"], r["model_name"], r["organization"], round(r["rating"], 1),
         ", ".join(difflib.get_close_matches(r["model_key"], list(ext), n=3, cutoff=0.8))) for r in latest]
display(spark.createDataFrame(rows, "model_key STRING, model_name STRING, organization STRING, rating DOUBLE, closest_candidates STRING"))

### CELL 8 - Optimization evidence for the report

Run it once after the first load. It prints the file layout of the largest fact table before and after `OPTIMIZE` (liquid clustering on `date_key` and `category_sk`), and the query plan of a typical dashboard filter. Take the Query Profile screenshot from the `display` of the last query (View query details, then Query profile).

In [ ]:
# CELL 8 - Optimization evidence
t = G("fact_arena_rating")
def detail():
    d = spark.sql(f"DESCRIBE DETAIL {t}").select("numFiles", "sizeInBytes", "clusteringColumns").collect()[0]
    return dict(files=d["numFiles"], size_mb=round(d["sizeInBytes"] / 1e6, 1), clustering=list(d["clusteringColumns"] or []))
print("before OPTIMIZE:", detail())
spark.sql(f"OPTIMIZE {t}")
print("after OPTIMIZE: ", detail())

probe = f"""SELECT m.display_name, f.rating, f.rating_rank
FROM {t} f JOIN {G('dim_model')} m ON f.model_sk = m.model_sk
JOIN {G('dim_arena_category')} c ON f.category_sk = c.category_sk
WHERE c.subset = 'text' AND c.category = 'overall' AND f.date_key >= 20260901
ORDER BY f.rating DESC LIMIT 20"""
spark.sql("EXPLAIN FORMATTED " + probe).show(truncate=False)
display(spark.sql(probe))

### CELL 9 - Health check

In [ ]:
# CELL 9 - Health check
display(spark.sql(" UNION ALL ".join(
    f"SELECT '{n}' AS table_name, '{TABLES[n]['kind']}' AS kind, COUNT(*) AS rows_loaded, MAX(load_timestamp) AS last_load FROM {G(n)}"
    for n in ORDER if n in TABLES)))
display(spark.sql(f"""SELECT layer, file_or_param, status, rows_read, rows_inserted, rows_updated, rows_skipped_duplicate,
    ROUND(duration_seconds, 1) AS seconds, start_time, message
    FROM {T_RUNS} WHERE layer LIKE 'Silver-to-Gold%' OR layer LIKE 'Gold%' ORDER BY load_timestamp DESC LIMIT 40"""))